# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

# Importation des données

# Utilisation des données lags 12

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

# On veut les séries stationnarisées pour le modèle
FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence (via UNRATE raw) pour avoir le vrai calendrier dispo
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

df_stationary["date"] = (
    pd.to_datetime(df_stationary["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
)

value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = (
    df_stationary[["series_id", "date", value_col]]
    .rename(columns={value_col: "value"})
)

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Dataset régression ciblé UNRATE
#    y = UNRATE (stationary)
#    exog = autres séries (contemporaines)
# ----------------------------
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)

df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x_long = df_x_long[df_x_long["date"].isin(df_y["date"])]

df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .reset_index()
)

df_model = (
    df_y[["date", "y"]]
    .merge(df_x, on="date", how="left")
    .dropna()
)

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
ts_lr shape: (788, 13)
Exog cols: ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


,unique_id,ds,y,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
0,UNRATE,1960-01-01,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
1,UNRATE,1960-02-01,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
2,UNRATE,1960-03-01,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
3,UNRATE,1960-04-01,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
4,UNRATE,1960-05-01,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


In [3]:
ts_lr["ds"] = (
    pd.to_datetime(ts_lr["ds"])
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# Dictionnaire de modèle

In [4]:
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

MLF_MODELS = {
    "LGBM_EXOG_ONLY": lambda freq, **params: MLForecast(
        models={"LGBM": LGBMRegressor(**params)},
        freq=freq,
        lags=[],               # EXOG-ONLY
        date_features=[],
    )
}

In [5]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error
from mlforecast.utils import PredictionIntervals

# ============================================================
# Helpers dates / fenêtres
# ============================================================
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

# ============================================================
# Tuning (random search) — param_grid fourni depuis le RUN
# ============================================================
def _tune_lgbm_on_train(
    ts_train,
    *,
    freq,
    h,
    tune_cv_windows,      # mini-CV pour le tuning (ex: 6)
    levels,
    param_grid,           # fourni depuis le RUN
    n_iter=100,           # fourni depuis le RUN
    min_train_n=None,
    seed=0,
    use_conformal_in_tune=False,  # recommandé False (beaucoup + rapide)
):
    """
    Tuning LightGBM sur train via mini Nixtla cross_validation (MAE).
    Renvoie (best_params, best_mae).
    """
    if min_train_n is not None and len(ts_train) < int(min_train_n):
        return None, np.nan

    best_params = None
    best_mae = np.inf

    # (optionnel) Conformal pendant le tuning (souvent inutile et coûteux)
    if use_conformal_in_tune:
        pi_tune = PredictionIntervals(h=h, n_windows=tune_cv_windows, method="conformal_distribution")
    else:
        pi_tune = None

    # ✅ Random search: n_iter essais au lieu de tout explorer
    for params in ParameterSampler(param_grid, n_iter=int(n_iter), random_state=seed):
        params = dict(params)

        # Defaults LGBM (stabilité)
        params.setdefault("random_state", seed)
        params.setdefault("verbosity", -1)
        params.setdefault("objective", "regression")
        params.setdefault("metric", "mae")
        params.setdefault("n_jobs", -1)
        params.setdefault("boosting_type", "gbdt")

        mlf = MLF_MODELS["LGBM_EXOG_ONLY"](freq, **params)

        cv = mlf.cross_validation(
            df=ts_train,
            h=h,
            step_size=1,
            n_windows=int(tune_cv_windows),
            prediction_intervals=pi_tune,            # None si use_conformal_in_tune=False
            level=list(levels) if pi_tune is not None else None,
            fitted=False,
            static_features=[],
            dropna=True,
        )

        mae = mean_absolute_error(cv["y"], cv["LGBM"])
        if mae < best_mae:
            best_mae = mae
            best_params = params

    return best_params, float(best_mae)

# ============================================================
# Backtesting mensuel (step=1), tuning tous les 36 mois
# param_grid + n_iter passés depuis le RUN
# ============================================================
def run_backtesting_h12_monthly_tune_every_36m_lgbm(
    ts,
    *,
    freq,
    h=12,
    exp_start="1990-01-01",
    exp_end="2025-08-01",
    step_size=1,
    pi_windows=24,
    levels=[95],
    tune_every_months=36,
    # tuning controls (depuis RUN)
    param_grid=None,
    tune_n_iter=100,
    tune_cv_windows=6,              # mini-CV pour tuning
    min_train_n=None,
    seed=0,
    use_conformal_in_tune=False,    # recommandé False
):
    if param_grid is None:
        raise ValueError("param_grid doit être fourni depuis le RUN (pas de grille par défaut ici).")

    ts = ts.copy()

    # dates MS
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    total_partitions = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # anti-fuite
    ts = ts[ts["ds"] <= exp_end].copy()

    # Conformal pour le backtest (ici oui)
    pi = PredictionIntervals(h=h, n_windows=pi_windows, method="conformal_distribution")

    # blocs (ex: 36 mois)
    blocks = []
    remaining = total_partitions
    cur_cutoff_start = cutoff_start_all
    while remaining > 0:
        n_win = min(int(tune_every_months), remaining)
        blocks.append((cur_cutoff_start, n_win))
        cur_cutoff_start = cur_cutoff_start + relativedelta(months=n_win)
        remaining -= n_win

    all_bkts = []
    params_history = []
    tune_mae_history = []

    for block_idx, (cutoff_start_blk, n_windows_blk) in enumerate(blocks, start=1):
        # données nécessaires au bloc (jusqu'au dernier ds prédit)
        ts_blk, cutoff_end_blk, ds_end_blk = _slice_cv_block(ts, cutoff_start_blk, n_windows_blk, h)

        # train pour tuning (<= cutoff_start_blk)
        ts_train_for_tune = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train_for_tune) < int(min_train_n):
            continue

        # tuning (random)
        best_params, tune_mae = _tune_lgbm_on_train(
            ts_train_for_tune,
            freq=freq,
            h=h,
            tune_cv_windows=min(int(tune_cv_windows), int(pi_windows)),
            levels=levels,
            param_grid=param_grid,
            n_iter=int(tune_n_iter),
            min_train_n=min_train_n,
            seed=seed,
            use_conformal_in_tune=use_conformal_in_tune,
        )
        if best_params is None:
            continue

        params_history.append({
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "n_windows": int(n_windows_blk),
            "tune_n_iter": int(tune_n_iter),
            "tune_cv_windows": int(min(int(tune_cv_windows), int(pi_windows))),
            **best_params,
        })
        tune_mae_history.append({
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "tune_mae": float(tune_mae),
        })

        # modèle figé sur le bloc
        mlf_blk = MLF_MODELS["LGBM_EXOG_ONLY"](freq, **best_params)

        bkt_blk = mlf_blk.cross_validation(
            df=ts_blk,
            h=h,
            step_size=int(step_size),
            n_windows=int(n_windows_blk),
            prediction_intervals=pi,
            level=list(levels),
            fitted=True,
            static_features=[],
            dropna=True,
        )

        bkt_blk["tune_block"] = block_idx
        bkt_blk["tune_mae"] = float(tune_mae)

        # stocker params principaux (debug)
        for k in [
            "subsample", "colsample_bytree", "num_leaves", "n_estimators", "max_depth",
            "reg_alpha", "reg_lambda", "min_child_samples", "min_split_gain"
        ]:
            if k in best_params:
                bkt_blk[k] = best_params[k]

        all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": "Aucun bloc backtest produit (min_train_n trop grand ?)"}

    bkt_df = pd.concat(all_bkts, ignore_index=True)

    # filtrer exactement l’expérience
    bkt_df = bkt_df[(bkt_df["ds"] >= exp_start) & (bkt_df["ds"] <= exp_end)].copy()
    bkt_df = bkt_df.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    meta = {
        "h": int(h),
        "step_size": int(step_size),
        "exp_start": exp_start,
        "exp_end": exp_end,
        "cutoff_start": cutoff_start_all,
        "cutoff_end": cutoff_end_all,
        "partitions": int(total_partitions),
        "pi_windows": int(pi_windows),
        "tune_every_months": int(tune_every_months),
        "tune_n_iter": int(tune_n_iter),
        "tune_cv_windows": int(tune_cv_windows),
        "use_conformal_in_tune": bool(use_conformal_in_tune),
        "param_grid": param_grid,
        "params_history": params_history,
        "tune_mae_history": tune_mae_history,
    }

    return bkt_df, meta

# Run 

In [7]:
# ============================================================
# RUN – LightGBM (Nixtla MLForecast) | EXOG-ONLY
# FIT chaque mois (step=1), tuning hyperparams tous les 36 mois
# horizon 12, bornes exactes exp
# + ds unique (dernier cutoff)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]

EXP_START = "1990-01-01"
EXP_END   = "2025-08-01"

TUNE_EVERY_MONTHS = 36

# ✅ Nouveaux paramètres tuning (random search)
TUNE_N_ITER = 100        # <- équivalent de n_iter=100
TUNE_CV_WINDOWS = 6      # <- mini CV pour comparer les configs
USE_CONFORMAL_IN_TUNE = False  # <- très recommandé (beaucoup + rapide)

# --------- Ta grille LightGBM (stabilité) ----------
PARAM_GRID = {
    "subsample":        [0.05,.1,.2,.3,.4,.5,.6,.7,.8,.9,1.0],
    "colsample_bytree": [.2,.3,.4,.5,.6,.7,1.0],
    "num_leaves":       [2,3,4,5,8,10,20,40,70,100],
    "n_estimators":     [5,10,20,30,40,50,75,100],
    "max_depth":        [1,2,3,5,8,15,-1],
    "reg_alpha":        [0, .1, 1, 2, 7, 10, 50, 100],
    "reg_lambda":       [0, .1, 1, 10, 20, 50, 100],
    "min_child_samples":[5,10,15],
    "min_split_gain":   [0.0, 0.01, 0.05],
}

# ts_lgbm: dataframe long Nixtla avec colonnes unique_id, ds, y + exog
ts_lgbm = ts_lr.copy()
ts_lgbm["ds"] = (
    pd.to_datetime(ts_lgbm["ds"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# anti-fuite
ts_lgbm = ts_lgbm[ts_lgbm["ds"] <= pd.Timestamp(EXP_END)].copy()

bkt_lgbm, meta_lgbm = run_backtesting_h12_monthly_tune_every_36m_lgbm(
    ts=ts_lgbm,
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    tune_every_months=TUNE_EVERY_MONTHS,
    param_grid=PARAM_GRID,             # <- grille fournie ici
    tune_n_iter=TUNE_N_ITER,           # ✅ nouveau
    tune_cv_windows=TUNE_CV_WINDOWS,   # ✅ nouveau
    use_conformal_in_tune=USE_CONFORMAL_IN_TUNE,  # ✅ nouveau
    min_train_n=36,
    seed=0,
)

print("✅ meta_lgbm keys:", list(meta_lgbm.keys()))
print("bkt_lgbm rows:", len(bkt_lgbm))

# ==========================
# ds unique: dernier cutoff
# ==========================
bkt_lgbm_final = (
    bkt_lgbm.sort_values(["unique_id", "ds", "cutoff"])
            .groupby(["unique_id", "ds"], as_index=False)
            .tail(1)
            .reset_index(drop=True)
)

print("bkt_lgbm_final rows        :", len(bkt_lgbm_final))
print("duplicates (unique_id, ds) :", bkt_lgbm_final.duplicated(["unique_id","ds"]).sum())
print("tune blocks (unique)       :", bkt_lgbm_final["tune_block"].nunique())

# quelques params utilisés
for col in ["num_leaves", "n_estimators", "max_depth", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"]:
    if col in bkt_lgbm_final.columns:
        print(f"{col} (unique):", sorted(bkt_lgbm_final[col].dropna().unique().tolist())[:12])

bkt_lgbm_final.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook
✅ meta_lgbm keys: ['h', 'step_size', 'exp_start', 'exp_end', 'cutoff_start', 'cutoff_end', 'partitions', 'pi_windows', 'tune_every_months', 'tune_n_iter', 'tune_cv_windows', 'use_conformal_in_tune', 'param_grid', 'params_history', 'tune_mae_history']
bkt_lgbm rows: 5070
bkt_lgbm_final rows        : 428
duplicates (unique_id, ds) : 0
tune blocks (unique)       : 12
num_leaves (unique): [5, 8, 20, 40, 70]
n_estimators (unique): [10, 40, 50, 100]
max_depth (unique): [1, 2, 3, 5, 8, 15]
subsample (unique): [0.1, 0.2, 0.6, 0.7, 0.8, 0.9, 1.0]
colsample_bytree (unique): [0.2, 0.3, 0.4, 0.6, 0.7, 1.0]
reg_alpha (unique): [0.0, 0.1, 1.0, 2.0, 10.0]
reg_lambda (unique): [0.0, 0.1, 1.0, 10.0, 50.0]


,unique_id,ds,cutoff,y,LGBM,LGBM-lo-95,LGBM-hi-95,tune_block,tune_mae,subsample,colsample_bytree,num_leaves,n_estimators,max_depth,reg_alpha,reg_lambda,min_child_samples,min_split_gain
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.204167,-0.557993,0.149660,1,0.267202,0.6,0.2,8,50,3,0.1,0.1,15,0.05
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.534207,-1.283757,0.215343,1,0.267202,0.6,0.2,8,50,3,0.1,0.1,15,0.05
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.582278,-1.786748,0.622192,1,0.267202,0.6,0.2,8,50,3,0.1,0.1,15,0.05
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.592693,-1.548685,0.363298,1,0.267202,0.6,0.2,8,50,3,0.1,0.1,15,0.05
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.053390,-0.703985,0.597206,1,0.267202,0.6,0.2,8,50,3,0.1,0.1,15,0.05


# Graphique

In [12]:
import pandas as pd
from utilsforecast.plotting import plot_series

# ============================================================
# 0) Safety checks
# ============================================================
df_plot = bkt_lgbm_final.copy()

# Dates
df_plot["ds"] = pd.to_datetime(df_plot["ds"], errors="coerce")
if df_plot["ds"].isna().any():
    bad = df_plot[df_plot["ds"].isna()].head()
    raise ValueError(f"Dates invalides dans bkt_lgbm_final['ds']:\n{bad}")

# Colonnes attendues
required = {"unique_id", "ds"}
missing = required - set(df_plot.columns)
if missing:
    raise ValueError(f"Colonnes manquantes dans bkt_lgbm_final: {missing}")

# y observé (selon sortie MLForecast c'est souvent 'y')
y_col = "y" if "y" in df_plot.columns else ("y_true" if "y_true" in df_plot.columns else None)
if y_col is None:
    raise ValueError("Je ne trouve pas la colonne des observations (attendu: 'y' ou 'y_true').")

# prédiction LGBM
pred_col = "LGBM" if "LGBM" in df_plot.columns else ("y_pred" if "y_pred" in df_plot.columns else None)
if pred_col is None:
    raise ValueError("Je ne trouve pas la colonne de prédiction (attendu: 'LGBM' ou 'y_pred').")

# bornes PI 95 : plusieurs conventions possibles
lo_candidates = [f"{pred_col}-lo-95", f"{pred_col}_lo_95", f"{pred_col}_lo", "LGBM-lo-95", "LGBM_lo_95"]
hi_candidates = [f"{pred_col}-hi-95", f"{pred_col}_hi_95", f"{pred_col}_hi", "LGBM-hi-95", "LGBM_hi_95"]

lo_col = next((c for c in lo_candidates if c in df_plot.columns), None)
hi_col = next((c for c in hi_candidates if c in df_plot.columns), None)

# ============================================================
# 1) Prepare obs
# ============================================================
df_obs = (
    df_plot.rename(columns={y_col: "y"})[["unique_id", "ds", "y"]]
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

# ============================================================
# 2) Prepare forecasts (+ PI si dispo)
# ============================================================
cols_fcst = ["unique_id", "ds", pred_col]
rename_fcst = {pred_col: "LGBM"}

if lo_col is not None and hi_col is not None:
    cols_fcst += [lo_col, hi_col]
    rename_fcst.update({lo_col: "LGBM-lo-95", hi_col: "LGBM-hi-95"})

df_fcst = (
    df_plot[cols_fcst]
    .rename(columns=rename_fcst)
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

# ============================================================
# 3) Plot
# ============================================================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95] if ("LGBM-lo-95" in df_fcst.columns and "LGBM-hi-95" in df_fcst.columns) else None,
    engine="plotly",
).update_layout(height=420)

# Renommer légendes
for tr in fig.data:
    if tr.name == "y":
        tr.name = "Unemployment rate (stationary)"
    elif tr.name == "LGBM":
        tr.name = "LightGBM (exog only)"
    elif "level_95" in (tr.name or "").lower():
        tr.name = "95% Prediction Interval"

fig.show()

# Evaluaer 

In [9]:
import numpy as np
import pandas as pd

# -----------------------------
# 1) One forecast per target date ds
#    (keep the most recent cutoff for each ds)
# -----------------------------
bkt_lgbm_final = (
    bkt_lgbm
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# 2) Segments + ALL
# -----------------------------
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-07-31", "2000-2008"),
    ("2008-08-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

def mae_safe(y_true, y_pred):
    df = pd.DataFrame({"y": y_true, "yhat": y_pred}).dropna()
    if len(df) == 0:
        return np.nan
    return float(np.mean(np.abs(df["y"].to_numpy() - df["yhat"].to_numpy())))

# Colonnes y / pred (selon sortie MLForecast)
y_col = "y" if "y" in bkt_lgbm_final.columns else ("y_true" if "y_true" in bkt_lgbm_final.columns else None)
p_col = "LGBM" if "LGBM" in bkt_lgbm_final.columns else ("y_pred" if "y_pred" in bkt_lgbm_final.columns else None)

if y_col is None:
    raise ValueError("Colonne des observations introuvable (attendu 'y' ou 'y_true').")
if p_col is None:
    raise ValueError("Colonne des prédictions introuvable (attendu 'LGBM' ou 'y_pred').")

rows = []

# --- ALL ---
rows.append({
    "period": "ALL",
    "n_obs": int(len(bkt_lgbm_final)),
    "MAE_LGBM": mae_safe(bkt_lgbm_final[y_col], bkt_lgbm_final[p_col]),
})

# --- Segments ---
for start, end, label in segments:
    mask = bkt_lgbm_final["ds"] >= pd.Timestamp(start)
    if end is not None:
        mask &= bkt_lgbm_final["ds"] <= pd.Timestamp(end)

    df_seg = bkt_lgbm_final.loc[mask]

    rows.append({
        "period": label,
        "n_obs": int(len(df_seg)),
        "MAE_LGBM": mae_safe(df_seg[y_col], df_seg[p_col]),
    })

df_mae_lgbm = pd.DataFrame(rows)

# Ordre propre
order = ["ALL"] + [s[2] for s in segments]
df_mae_lgbm["period"] = pd.Categorical(df_mae_lgbm["period"], categories=order, ordered=True)
df_mae_lgbm = df_mae_lgbm.sort_values("period").reset_index(drop=True)

df_mae_lgbm

,period,n_obs,MAE_LGBM
0,ALL,428,0.737209
1,1990-1999,120,0.478132
2,2000-2008,103,0.401694
3,2008-2019,137,0.679676
4,2020-fin,68,1.818520


Très bien partie

# Sauvegarde

In [ ]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (Linear Regression)
# ============================================================

from datetime import datetime
import json

# ----------------------------
# Model identification
# ----------------------------
SERIES_ID = "UNRATE"
MODEL_TAG = "lr_lag12_exog"   # 🔑 clair et extensible

# ----------------------------
# Directories
# ----------------------------
OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) Outputs (OOS forecasts)
# ----------------------------
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_oos_forecasts.parquet"
df_lr_forecasts.to_parquet(oos_path, index=False)

# ----------------------------
# 2) Artifacts – raw backtesting output
# ----------------------------
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_bkt_raw.parquet"
bkt_lr.to_parquet(bkt_path, index=False)

# ----------------------------
# 3) Run configuration (reproducibility)
# ----------------------------
run_config = {
    "model": "LinearRegression",
    "framework": "Nixtla-MLForecast",
    "target": SERIES_ID,
    "stationary": True,
    "lags": [12],
    "exogenous_variables": [
        c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]
    ],
    "horizon": H,
    "step_size": STEP_SIZE,
    "partitions": PARTITIONS,
    "prediction_intervals": {
        "method": "conformal_distribution",
        "levels": LEVELS,
        "n_windows": PI_WINDOWS,
    },
    "frequency": FREQ,
}

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ----------------------------
# 4) Metadata – run info
# ----------------------------
meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "model_tag": MODEL_TAG,
    "files": {
        "oos_forecasts": str(oos_path.resolve()),
        "cv_raw": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved:")
print(" - OOS forecasts :", oos_path.name)
print(" - CV raw        :", bkt_path.name)
print(" - Config        :", cfg_path.name)
print(" - Metadata      :", meta_path.name)


✅ Saved:
 - OOS forecasts : unrate_lr_lag12_exog_oos_forecasts.parquet
 - CV raw        : unrate_lr_lag12_exog_bkt_raw.parquet
 - Config        : unrate_lr_lag12_exog_config.json
 - Metadata      : unrate_lr_lag12_exog_run_info.json


C:\Users\Mita\AppData\Local\Temp\ipykernel_6056\677556810.py:72: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



# Graphique

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 